# PharmaGuard — EDA & Model Training
**Dataset**: NASA CMAPSS Turbofan Engine Degradation (FD001)

This notebook covers:
1. Dataset loading & structure
2. Exploratory Data Analysis (sensor distributions, correlation)
3. Feature engineering (RUL labeling, rolling window stats)
4. Model training: Random Forest (RUL) + Isolation Forest (anomaly)
5. Evaluation & visualisation
6. Export for FastAPI

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

plt.style.use('dark_background')
TEAL = '#00D4AA'
print('Dependencies loaded ✅')

## 1. Load CMAPSS FD001

In [ ]:
DATA_DIR = '../backend/data'

COLS = ['unit','cycle','op1','op2','op3'] + [f's{i}' for i in range(1,22)]
DROP = ['s1','s5','s6','s10','s16','s18','s19']  # near-zero variance on FD001
RUL_CAP = 125

def load(split):
    df = pd.read_csv(f'{DATA_DIR}/{split}_FD001.txt', sep=r'\s+', header=None, names=COLS)
    return df.drop(columns=DROP)

train_df = load('train')
test_df  = load('test')
rul_df   = pd.read_csv(f'{DATA_DIR}/RUL_FD001.txt', header=None, names=['RUL_true'])

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()

## 2. EDA — Sensor Distributions

In [ ]:
sensor_cols = [c for c in train_df.columns if c.startswith('s')]

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for ax, col in zip(axes.flat, sensor_cols):
    ax.hist(train_df[col], bins=50, color=TEAL, alpha=0.7, edgecolor='none')
    ax.set_title(col, fontsize=9)
    ax.set_yticks([])
plt.suptitle('CMAPSS FD001 — Sensor Distributions', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## 3. Feature Engineering — RUL Labels & Rolling Stats

In [ ]:
def add_rul(df):
    max_c = df.groupby('unit')['cycle'].max().rename('max_cycle')
    df = df.join(max_c, on='unit')
    df['RUL'] = (df['max_cycle'] - df['cycle']).clip(upper=RUL_CAP)
    return df.drop(columns=['max_cycle'])

WINDOW = 10
def add_rolling(df, cols):
    grp = df.groupby('unit')[cols]
    for col in cols:
        df[f'{col}_mean{WINDOW}'] = grp[col].transform(lambda x: x.rolling(WINDOW, min_periods=1).mean())
        df[f'{col}_std{WINDOW}']  = grp[col].transform(lambda x: x.rolling(WINDOW, min_periods=1).std().fillna(0))
    return df

train_df = add_rul(train_df)
train_df = add_rolling(train_df, sensor_cols)

print(f'Feature count after engineering: {train_df.shape[1]}')

# Plot RUL distribution
plt.figure(figsize=(10, 4))
plt.hist(train_df['RUL'], bins=60, color=TEAL, alpha=0.8, edgecolor='none')
plt.title('RUL Distribution (capped at 125 cycles)')
plt.xlabel('Remaining Useful Life (cycles)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Model Training

In [ ]:
feature_cols = (['op1','op2','op3'] + sensor_cols +
    [c for c in train_df.columns if '_mean' in c or '_std' in c])

X = train_df[feature_cols].values
y = train_df['RUL'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42)

scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

print(f'Training samples: {len(X_train):,}  |  Validation: {len(X_val):,}')

rf = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_split=5, n_jobs=-1, random_state=42)
rf.fit(X_train_s, y_train)

y_pred = rf.predict(X_val_s)
print(f'RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.2f}')
print(f'MAE:  {mean_absolute_error(y_val, y_pred):.2f}')
print(f'R²:   {r2_score(y_val, y_pred):.4f}')

In [ ]:
# Isolation Forest for anomaly detection
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(X_train_s)
print('IsolationForest trained ✅')

## 5. Evaluation Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs Actual
ax = axes[0]
ax.scatter(y_val[:2000], y_pred[:2000], alpha=0.3, s=4, color=TEAL)
ax.plot([0, RUL_CAP], [0, RUL_CAP], 'r--', linewidth=1, label='Perfect fit')
ax.set_xlabel('Actual RUL')
ax.set_ylabel('Predicted RUL')
ax.set_title('Predicted vs Actual RUL')
ax.legend()

# Residuals
ax = axes[1]
residuals = y_pred - y_val
ax.hist(residuals, bins=60, color='#60A5FA', alpha=0.7, edgecolor='none')
ax.axvline(0, color='red', linestyle='--')
ax.set_xlabel('Residual (predicted − actual)')
ax.set_title('Residual Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
imps = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)[:15]

plt.figure(figsize=(10, 5))
imps.plot(kind='barh', color=TEAL, alpha=0.85)
plt.gca().invert_yaxis()
plt.title('Top 15 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
bundle = {
    'rf_model':            rf,
    'iso_model':           iso,
    'scaler':              scaler,
    'feature_cols':        feature_cols,
    'rul_cap':             RUL_CAP,
    'metrics':             {'rmse': float(np.sqrt(mean_squared_error(y_val, y_pred))),
                            'mae':  float(mean_absolute_error(y_val, y_pred)),
                            'r2':   float(r2_score(y_val, y_pred))},
    'feature_importances': dict(zip(feature_cols, rf.feature_importances_.tolist())),
}

out_path = '../backend/model/model.pkl'
joblib.dump(bundle, out_path)
print(f'Model saved → {out_path} ✅')